# روش‌های رسمی، تیم ۳
بررسی سازگاری نیازمندی‌های زمان‌حقیقی و استخراج MIS با Python/Z3. محیط Colab را روی CPU بگذارید. این نوت‌بوک فایل‌های مخزن را اجرا می‌کند.
در GitHub، نسخهٔ پیش‌فرض از شاخهٔ `final-project-completion` دریافت می‌شود. برای ZIP، پیش از سلول اول مسیر پوشهٔ استخراج‌شده را در `TEAM3_PROJECT` تنظیم کنید.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import tempfile

remote = "https://github.com/Ehsan-Rezaeii/realtime-requirements-mis.git"
release_ref = "final-project-completion"
requested_ref = os.environ.get("TEAM3_REF", "").strip()
ref = requested_ref or release_ref
project_setting = os.environ.get("TEAM3_PROJECT")
source = Path(project_setting).expanduser().resolve() if project_setting else None
runtime_root = Path("/content") if Path("/content").is_dir() else Path(tempfile.gettempdir())

def git(*args, cwd=None):
    return subprocess.check_output(["git", *args], cwd=cwd, text=True).strip()

if source is not None and not (source / ".git").exists():
    if requested_ref:
        raise ValueError("For an extracted ZIP, unset TEAM3_REF. To select a Git version, unset TEAM3_PROJECT.")
    project = source
else:
    # Manage a separate Git cache. Even a dirty user checkout is left intact.
    cache = runtime_root / ".team3-release"
    cache.mkdir(exist_ok=True)
    repo = cache / "repository"
    if not repo.exists():
        git("clone", "--no-checkout", remote, str(repo))
    if git("remote", "get-url", "origin", cwd=repo) != remote:
        raise RuntimeError("The notebook Git cache points to a different repository.")
    if ref.startswith("-") or ":" in ref:
        raise ValueError("TEAM3_REF must be a remote branch, tag or commit.")
    git("fetch", "--no-tags", "origin", ref, cwd=repo)
    commit = git("rev-parse", "--verify", "FETCH_HEAD^{commit}", cwd=repo)
    project = cache / commit
    if project.exists():
        if git("rev-parse", "HEAD", cwd=project) != commit or git("status", "--porcelain", cwd=project):
            project = Path(tempfile.mkdtemp(prefix=f"{commit[:12]}-", dir=cache))
    if not (project / ".git").exists():
        git("worktree", "add", "--detach", str(project), commit, cwd=repo)
    print(f"Git commit: {commit}")

if not (project / "setup.sh").is_file():
    raise RuntimeError("Set TEAM3_PROJECT to the project directory inside the extracted ZIP.")

os.environ["PYTHON"] = sys.executable
print(project)

In [ ]:
def run(*args):
    subprocess.run(args, cwd=project, check=True)

run("bash", "setup.sh")
run(sys.executable, "-c", "import sys,z3; print(sys.version); print(z3.get_version_string())")
run("bash", "scripts/run_project.sh", "demo")
run("bash", "scripts/run_project.sh", "test")

## محل ذخیره و اجرای پایلوت
در Colab می‌توانید Google Drive را mount کنید. مطالعه با همان مسیر و مشخصات محیط ادامه می‌یابد. اگر محیط عوض شد، یک مسیر تازه برای اجرای قابل مقایسه انتخاب کنید.

In [ ]:
SAVE_TO_DRIVE = False
output_root = project / "results" / "notebook"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    output_root = Path("/content/drive/MyDrive/team3-formal-methods")
pilot = output_root / "pilot"
run("bash", "scripts/run_project.sh", "pilot", str(pilot))

## اجرای اصلی
اجرای کامل چند دقیقه زمان می‌برد. برای شروع یا ادامه مقدار `RUN_FULL` را True کنید.

In [ ]:
RUN_FULL = False
final = output_root / "final"
if RUN_FULL:
    run("bash", "scripts/run_project.sh", "full", str(final))
evidence = final if (final / "summary").is_dir() else pilot
run(sys.executable, "analysis/analyze_results.py", "--study", str(evidence))

In [ ]:
from IPython.display import Image, display
import pandas as pd
display(pd.read_csv(evidence / "summary/run_status.csv"))
for image in sorted((evidence / "figures").glob("*.png")):
    display(Image(filename=str(image)))

In [ ]:
import shutil
bundle = shutil.make_archive(str(output_root / (evidence.name + "-evidence")), "zip", evidence)
print(bundle)
DOWNLOAD_IN_COLAB = False
if DOWNLOAD_IN_COLAB:
    from google.colab import files
    files.download(bundle)